# 22. Exercise Definition Test (Pipeline 03)

- Goal: check selected exercise-definition loading, split YAML resolution, and pipeline Step 3 integration.
- Docs: docs_eng/pipeline/03_exercise_definition.md / docs/pipeline/03_exercise_definition.md
- Inputs: canonical or authoring-draft exercise definition YAML files, plus p01 pose/annotation CSV for the integration check.
- Outputs: selected definition summary and exercise_definition report from run_pipeline().
- Verification points: selected definition is not generic fallback; key fields are populated; camera protocol matches recording metadata when available; pipeline report contains exercise-definition provenance.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import warnings

from movement.exercise_definition import load_exercise_definition
from movement.stage_context import find_project_root, resolve_target_definitions_dir

PROJECT_ROOT = find_project_root()

# Default stage-check target. Change this value only when testing another exercise definition.
TARGET_EXERCISE_ID = "squat"
TARGET_DEFINITIONS_DIR = resolve_target_definitions_dir(
    TARGET_EXERCISE_ID,
    project_root=PROJECT_ROOT,
)


## Case 1: Load Selected Definition

For the current p01 stage-check path, load one selected definition first.
The primary selector is `exercise_id`; legacy annotation files that only provide `exercise_type` are treated as aliases.
When a non-runtime authoring exercise id is selected, the notebook resolves the matching local draft bundle before the git-tracked example bundle.
Broader registry coverage belongs in unit tests or a separate registry audit.


In [ ]:
exercise_def = load_exercise_definition(TARGET_EXERCISE_ID, TARGET_DEFINITIONS_DIR)
selected_defs = {exercise_def.exercise_id: exercise_def}

print("loaded selected definition:")
print(f"  exercise_id     : {exercise_def.exercise_id}")
print(f"  version         : {exercise_def.version}")
print(f"  fallback        : {exercise_def.is_generic_fallback}")
print(f"  definitions_dir : {TARGET_DEFINITIONS_DIR}")

In [ ]:
import pandas as pd

expected_path = TARGET_DEFINITIONS_DIR / f"{TARGET_EXERCISE_ID}.yaml"
assert expected_path.exists(), f"missing selected definition YAML: {expected_path}"
assert exercise_def.exercise_id == TARGET_EXERCISE_ID
assert exercise_def.is_generic_fallback is False

summary = pd.DataFrame([
    {
        'exercise_id': exercise_def.exercise_id,
        'display_name': exercise_def.display_name,
        'definitions_dir': str(TARGET_DEFINITIONS_DIR),
        'is_generic_fallback': exercise_def.is_generic_fallback,
        'laterality': exercise_def.classification.get('laterality'),
        'posture_type': exercise_def.classification.get('posture_type'),
    }
])
display(summary)
print("PASS: selected exercise definition loaded for the current stage-check target")


## Case 2: Selected Definition Field Inspection

Spot-check the most important typed fields for the selected definition.

In [ ]:
for ex_id, ed in selected_defs.items():
    clf = ed.classification
    print(f"── {ex_id} ──────────────────────────────")
    print(f"  laterality       : {clf.get('laterality')}")
    print(f"  posture_type     : {clf.get('posture_type')}")
    print(f"  primary_plane    : {clf.get('primary_plane')}")
    print(f"  phase_model.type : {ed.phase_model.type}")
    print(f"  expected_ratio   : {ed.phase_model.expected_ratio}")
    print(f"  primary_joints   : {ed.landmarks.primary_joints}")
    print(f"  compensation_patterns ({len(ed.compensation_patterns)}): {ed.compensation_patterns}")
    print()

## Case 3: Pipeline Step Integration

Run the pipeline through Step 3 with the selected definition and confirm the exercise-definition report.


In [ ]:
from movement.annotation import load_annotation_csv
from movement.config import LANDMARKS
from movement.io import load_pose_csv
from movement.pipeline import load_pipeline_config, run_pipeline
from movement.stage_context import find_project_root

PROJECT_ROOT = find_project_root()
config_path = PROJECT_ROOT / "configs/pipeline_default.yaml"
csv_path = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv"
ann_path = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_annotation.csv"

config = load_pipeline_config(config_path)
df = load_pose_csv(csv_path)
ann_df = load_annotation_csv(ann_path)

# Step 3 normally receives exercise context from Step 2 annotation when
# exercise_id is not explicitly configured. Enable annotation here so this
# stage check validates that handoff instead of the generic fallback path.
config.annotation.enabled = True
config.annotation.path = ann_path

selected_exercise_id = globals().get("TARGET_EXERCISE_ID", "squat")
selected_definitions_dir = globals().get(
    "TARGET_DEFINITIONS_DIR",
    PROJECT_ROOT / "data/definitions/exercises",
)

# If Case 1 was switched to a draft or another selected definition, run the
# pipeline against that explicit definition. With the default squat target,
# keep the annotation-derived handoff path under test.
if selected_exercise_id != "squat":
    config.exercise_definition.exercise_id = selected_exercise_id
    config.exercise_definition.definitions_dir = str(selected_definitions_dir)

print("annotation.enabled:", config.annotation.enabled)
print("annotation.path:", config.annotation.path)
print("selected_exercise_id:", selected_exercise_id)
print("selected_definitions_dir:", selected_definitions_dir)
print("exercise_definition.enabled:", config.exercise_definition.enabled)
print("exercise_definition.definitions_dir:", config.exercise_definition.definitions_dir)
print("exercise_definition.exercise_id:", config.exercise_definition.exercise_id)


In [ ]:
import warnings as _w

with _w.catch_warnings(record=True) as caught:
    _w.simplefilter("always")
    result_df, report = run_pipeline(df, config=config, landmarks=LANDMARKS, ann_df=ann_df)

print("steps executed:", list(report.keys()))
print()

if caught:
    print(f"{len(caught)} warning(s) during pipeline run:")
    for w in caught:
        print(f"  [{w.category.__name__}] {w.message}")

In [ ]:
import json

assert "exercise_definition" in report, "exercise_definition step missing from report"
exd_report = report["exercise_definition"]
print(json.dumps(exd_report, indent=2))

annotation_id_column = "exercise_id" if "exercise_id" in ann_df.columns else "exercise_type"
annotated_exercise_ids = sorted(
    str(value) for value in ann_df[annotation_id_column].dropna().unique()
)
configured_exercise_id = config.exercise_definition.exercise_id
configured_definitions_dir = Path(config.exercise_definition.definitions_dir)
if not configured_definitions_dir.is_absolute():
    configured_definitions_dir = PROJECT_ROOT / configured_definitions_dir
known_definition_ids = {path.stem for path in configured_definitions_dir.glob("*.yaml")}

if configured_exercise_id:
    if configured_exercise_id in known_definition_ids:
        assert exd_report["exercise_id"] == configured_exercise_id
        if configured_exercise_id != "generic":
            assert exd_report["is_generic_fallback"] is False
    else:
        assert exd_report["exercise_id"] == "generic"
        assert exd_report["is_generic_fallback"] is True
else:
    expected_specific_ids = [
        exercise_id
        for exercise_id in annotated_exercise_ids
        if exercise_id in known_definition_ids
    ]
    if expected_specific_ids:
        assert exd_report["exercise_id"] in expected_specific_ids, (
            "exercise_definition did not load the annotated exercise_id: "
            f"annotated={expected_specific_ids}, loaded={exd_report['exercise_id']}"
        )
        assert exd_report["is_generic_fallback"] is False
    else:
        assert exd_report["is_generic_fallback"] is True

print()
print("PASS: exercise_definition step in report")
print(f"  configured exercise_id        : {configured_exercise_id}")
print(f"  configured definitions_dir    : {configured_definitions_dir}")
print(f"  annotated id column           : {annotation_id_column}")
print(f"  annotated exercise_id values  : {annotated_exercise_ids}")
print(f"  exercise_id                   : {exd_report['exercise_id']}")
print(f"  movement_template_id          : {exd_report.get('movement_template_id')}")
print(f"  is_generic_fallback           : {exd_report['is_generic_fallback']}")


## Check Summary

This notebook is a compact execution/QC checkpoint for the selected exercise definition. Negative loader fixtures and fallback behavior are covered by pytest, not this user-facing stage check.
